# Learning Python's `asyncio` Module

`asyncio` is Python's library for writing concurrent programs with `async` and `await`. It is especially useful for I/O-bound work: network requests, timers, sockets, and other operations that spend time waiting.

Concurrency is not the same as parallel execution. An asyncio event loop switches between tasks when they await; it does not make ordinary Python CPU-heavy work run on multiple cores.

## Core ideas

- A **coroutine function** is declared with `async def`.
- Calling a coroutine function creates a **coroutine object**; it does not run the function yet.
- `await` pauses the current coroutine until an awaitable finishes, allowing other tasks to run.
- A **Task** schedules a coroutine to run on the event loop.
- The **event loop** coordinates tasks and resumes them when awaited work is ready.

This notebook runs code in Jupyter, so examples use top-level `await`. In a regular `.py` script, start the event loop with `asyncio.run(main())`.

In [2]:
import asyncio


async def greet():
    print('Coroutine started')
    await asyncio.sleep(0.2)
    print('Coroutine resumed after waiting')
    return 'Coroutine finished'


result = await greet()
print(result)

Coroutine started
Coroutine resumed after waiting
Coroutine finished


## Sequential work vs concurrency

If each operation is awaited before starting the next, they run sequentially. `asyncio.gather()` lets several awaitables make progress concurrently. `asyncio.create_task()` schedules a coroutine as a Task so it can run while the current coroutine does other work.

`asyncio.sleep()` pauses without blocking the event loop. By contrast, `time.sleep()` blocks the whole thread and prevents other asyncio tasks from running.

In the next cell, simulated I/O takes about 0.4 seconds sequentially and about 0.2 seconds concurrently. Actual timings vary by machine.

In [3]:
import asyncio
import time


async def simulated_io(label, delay):
    print(f'Starting {label}')
    await asyncio.sleep(delay)
    print(f'Finished {label}')
    return label


async def run_sequentially():
    start = time.perf_counter()
    first = await simulated_io('first', 0.2)
    second = await simulated_io('second', 0.2)
    print('Sequential results:', [first, second])
    print(f'Sequential duration: {time.perf_counter() - start:.2f} seconds')


async def run_concurrently():
    start = time.perf_counter()
    first_task = asyncio.create_task(simulated_io('first', 0.2))
    second_task = asyncio.create_task(simulated_io('second', 0.2))
    results = await asyncio.gather(first_task, second_task)
    print('Concurrent results:', results)
    print(f'Concurrent duration: {time.perf_counter() - start:.2f} seconds')


await run_sequentially()
await run_concurrently()

Starting first
Finished first
Starting second
Finished second
Sequential results: ['first', 'second']
Sequential duration: 0.41 seconds
Starting first
Starting second
Finished first
Finished second
Concurrent results: ['first', 'second']
Concurrent duration: 0.22 seconds


## Common `asyncio` functions and keywords

| Name | Purpose |
| --- | --- |
| `async def` | Defines a coroutine function. Calling it returns a coroutine object; it needs to be awaited or scheduled. |
| `await` | Waits for a coroutine or other awaitable and gives the event loop a chance to run other tasks. |
| `asyncio.run(main())` | Starts and closes an event loop for a normal Python script. In this notebook, use top-level `await main()` instead. |
| `asyncio.sleep(seconds)` | Waits without blocking the event loop; useful for timers and demonstrations. |
| `asyncio.create_task(coro)` | Schedules a coroutine to run concurrently and returns a Task. |
| `asyncio.gather(*awaitables)` | Runs awaitables concurrently and returns their results in input order. |
| `asyncio.TaskGroup()` | Groups related tasks and waits for them together. If a task fails, the group cancels its remaining tasks. Available in Python 3.11+. |
| `asyncio.wait_for(awaitable, timeout=...)` | Stops waiting after a deadline and raises `TimeoutError`. |
| `asyncio.timeout(seconds)` | Async context manager that applies a timeout to a block. Available in Python 3.11+. |
| `asyncio.to_thread(function, *args)` | Runs a blocking synchronous function in a worker thread so it does not block the event loop. |
| `async for` / `async with` | Consume an asynchronous iterator or enter an asynchronous context manager. |

`TaskGroup` is a structured way to manage related tasks. It waits for all child tasks before leaving the `async with` block.

In [4]:
import asyncio


async def load_data(label, delay):
    await asyncio.sleep(delay)
    return f'{label} is ready'


async def main():
    async with asyncio.TaskGroup() as task_group:
        profile_task = task_group.create_task(load_data('Profile', 0.2))
        messages_task = task_group.create_task(load_data('Messages', 0.1))

    print(profile_task.result())
    print(messages_task.result())


await main()

Profile is ready
Messages is ready


## Timeouts and blocking functions

Use a timeout when an operation should not wait forever. `asyncio.wait_for()` cancels the awaited operation when its deadline expires and raises `TimeoutError`. `asyncio.timeout()` offers the same idea as an `async with` block.

A normal blocking function such as `time.sleep()` should not run directly inside an async function: it freezes the event-loop thread. `asyncio.to_thread()` can move blocking I/O or other suitable synchronous work to a worker thread. It is not a general way to speed up CPU-heavy Python code.

In [5]:
import asyncio
import time


def blocking_lookup():
    time.sleep(0.2)
    return 'Blocking lookup finished'


async def main():
    try:
        await asyncio.wait_for(asyncio.sleep(1), timeout=0.1)
    except TimeoutError:
        print('The operation timed out')

    lookup_task = asyncio.to_thread(blocking_lookup)
    heartbeat_task = asyncio.sleep(0.05, result='Other async work continued')
    results = await asyncio.gather(lookup_task, heartbeat_task)
    print(results)


await main()

The operation timed out
['Blocking lookup finished', 'Other async work continued']


## Async iterators and context managers

An **async generator** is an `async def` function that uses `yield`. Consume it with `async for`; each next value may require waiting. Use `async with` for an asynchronous context manager that needs to await setup or cleanup, such as opening an async network connection.

| Syntax | Meaning |
| --- | --- |
| `async def` with `yield` | Produces values over time as an async generator. |
| `async for value in source` | Iterates over an asynchronous iterable, awaiting each item. |
| `async with resource` | Awaits asynchronous resource setup and cleanup. |

The next example uses `@asynccontextmanager` to create a small asynchronous resource manager.

In [6]:
import asyncio
from contextlib import asynccontextmanager


async def countdown():
    for number in range(3, 0, -1):
        await asyncio.sleep(0.05)
        yield number


@asynccontextmanager
async def open_resource():
    print('Opening resource')
    try:
        yield 'example connection'
    finally:
        print('Closing resource')


async def main():
    async for number in countdown():
        print('Countdown:', number)

    async with open_resource() as resource:
        print('Using', resource)


await main()

Countdown: 3
Countdown: 2
Countdown: 1
Opening resource
Using example connection
Closing resource


## Coordinating multiple tasks

These helpers coordinate tasks that share work or resources:

- `asyncio.Queue`: passes items safely between producer and consumer tasks. Common methods are `put()`, `get()`, `task_done()`, and `join()`.
- `asyncio.Lock`: allows only one task at a time into a protected section. Use `async with lock:` so it is released even if an exception occurs.
- `asyncio.Semaphore(limit)`: allows at most `limit` tasks into a section at once. Useful for limiting simultaneous requests.
- `asyncio.Event`: lets one task signal other tasks with `set()`; waiting tasks use `await event.wait()`.

This example uses a semaphore to limit active workers to two.

In [7]:
import asyncio


async def process_item(item, semaphore):
    async with semaphore:
        print(f'Starting item {item}')
        await asyncio.sleep(0.1)
        print(f'Finished item {item}')


async def main():
    semaphore = asyncio.Semaphore(2)
    await asyncio.gather(
        *(process_item(item, semaphore) for item in range(1, 5))
    )


await main()

Starting item 1
Starting item 2
Finished item 1
Finished item 2
Starting item 3
Starting item 4
Finished item 3
Finished item 4


## Common mistakes and practice

- Calling an `async def` function without `await` or scheduling it creates a coroutine that does not run.
- `await` only works inside an `async def` function, except in notebook cells, which support top-level `await`.
- Do not call `asyncio.run()` inside a notebook or inside code that already has a running event loop. Use it once at the entry point of a regular script.
- Do not use blocking operations such as `time.sleep()` directly in an async function. Use `await asyncio.sleep()` for a timer, or `asyncio.to_thread()` for suitable blocking functions.
- Asyncio is most helpful when tasks spend time waiting. CPU-heavy pure-Python work generally needs a process pool or another parallel-computing approach.

### Try it yourself

1. Change the delays in the sequential/concurrent example and compare the timings.
2. Add a third task to the `TaskGroup` example.
3. Change the semaphore limit from `2` to `1` and observe the worker order.
4. Add a queue producer and consumer using `asyncio.Queue`, `put()`, `get()`, and `task_done()`.

For reference, `asyncio` provides many more APIs for streams, subprocesses, synchronization, and networking. Start with the patterns in this notebook and consult the official Python `asyncio` documentation as needed.